## CodebookQA + TRL pipeline demo (SFT + GRPO)

This notebook demonstrates the features added/refactored today:

- **Content-only datasets**: JSONL rows store only `Story/Codebook/Question/Assistant...` (no long instruction prefix).
- **Three prompt styles** everywhere:
  - **full**: long instruction prefix
  - **abbr**: fixed short tag `TASK: CODEBOOK_QA\n\n`
  - **none**: no prefix
- **Two SFT data sources**:
  - **jsonl**: train from `data/*.jsonl`
  - **live**: train from on-the-fly `CodebookQADataset` via `dataloader/trl_adapters.py`
- **GRPO training**: train from live dataloader with configurable prompt style.

Notes:
- Some steps require packages like `datasets`, `trl`, `torch`, and model weights. This notebook focuses on *how to call the pipeline*.


### 1) Prompt styles (full / abbr / none)

The shared prompt-style logic lives in `dataloader/trl_adapters.py`:

- `build_task_prefix(style)` where `style in {"full","abbr","none"}`
- `DEFAULT_ABBR_PREFIX` is fixed to:

```text
TASK: CODEBOOK_QA

```


In [ ]:
from dataloader.trl_adapters import build_task_prefix, DEFAULT_ABBR_PREFIX

print("DEFAULT_ABBR_PREFIX repr:", repr(DEFAULT_ABBR_PREFIX))

for style in ["none", "abbr", "full"]:
    p = build_task_prefix(style)
    print("\nstyle:", style)
    print(p[:400] + ("..." if len(p) > 400 else ""))


### 2) Content-only JSONL datasets

Your SFT JSONL files are now expected to store **content-only** `text`, starting at `Story:\n`.

- Example file: `data/codebook_qa_sft_gpt_1000.jsonl` (after stripping)
- Deterministic file: `data/codebook_qa_sft_deterministic_1000.jsonl`

At training time you choose whether to prepend `full` / `abbr` / `none`.


In [ ]:
import json
from pathlib import Path

path = Path("data/codebook_qa_sft_gpt_1000.jsonl")
first = path.read_text(encoding="utf-8").splitlines()[0]
obj = json.loads(first)
print("keys:", sorted(obj.keys()))
print("text starts with:", repr(obj["text"][:20]))
print("contains Story marker:", "Story:\n" in obj["text"])

### 3) Generating JSONL datasets

#### 3a) GPT teacher SFT data (content-only `text`)

This script prompts the teacher with a **full** prefix, but saves `text` as **content-only**:

- `scripts/annotation/annotate_codebook_qa_sft.py`

Example command:

```bash
python3 scripts/annotation/annotate_codebook_qa_sft.py \
  --num-examples 1000 \
  --output data/codebook_qa_sft_gpt_1000.jsonl \
  --split train
```

#### 3b) Deterministic SFT data (content-only `text`)

- `scripts/annotation/annotate_codebook_qa_sft_deterministic.py`

Example command:

```bash
python3 scripts/annotation/annotate_codebook_qa_sft_deterministic.py \
  --num-examples 1000 \
  --output data/codebook_qa_sft_deterministic_1000.jsonl \
  --split train
```


### 4) Stripping a long prefix from older JSONL files

If you have an older JSONL where `text` starts with a long instruction prefix, you can normalize it to content-only using:

- `scripts/strip_jsonl_prefix.py`

Example:

```bash
python3 scripts/strip_jsonl_prefix.py \
  --input data/codebook_qa_sft_gpt_older.jsonl \
  --output data/codebook_qa_sft_gpt_older_noprefix.jsonl
```

Rule: the script keeps everything from the first `Story:\n` onward.


### 5) Using the live dataloader + adapters

The live dataset generates graphs/codebooks/questions on the fly:

- `dataloader/codebook_qa.py`: `CodebookQADataset`
- `dataloader/trl_adapters.py`: `CodebookQASFTDataset` and `CodebookQAGRPODataset`

Below we construct small datasets and inspect one example for each prompt style.


In [ ]:
from dataloader import CodebookQADataset, GraphDifficultyConfig
from dataloader.trl_adapters import CodebookQASFTDataset, CodebookQAGRPODataset, SFTAdapterConfig

base = CodebookQADataset(
    split="train",
    difficulties=[(GraphDifficultyConfig(goal_depth=2, max_leaf_nodes=4), 1.0)],
    seed=0,
)

for style in ["none", "abbr", "full"]:
    sft = CodebookQASFTDataset(
        base_dataset=base,
        num_examples=1,
        config=SFTAdapterConfig(prompt_style=style),
    )
    print("\nSFT style:", style)
    print(sft[0]["text"][:400] + "...\n")

for style in ["none", "abbr", "full"]:
    grpo = CodebookQAGRPODataset(
        base_dataset=base,
        num_examples=1,
        prompt_style=style,
    )
    print("GRPO style:", style)
    print(grpo[0]["prompt"][:250] + "...\n")


### 6) SFT training (TRL)

`scripts/train_sft_codebook_qa.py` supports both snapshot JSONL and live dataloader.

#### 6a) Train SFT from JSONL (content-only) and choose prefix at load time

```bash
python3 scripts/train_sft_codebook_qa.py \
  --data-source jsonl \
  --data-path data/codebook_qa_sft_gpt_1000.jsonl \
  --prompt-style abbr \
  --model Qwen/Qwen2-0.5B-Instruct \
  --output-dir Qwen2-CodebookQA-SFT
```

#### 6b) Train SFT from the live dataloader

```bash
python3 scripts/train_sft_codebook_qa.py \
  --data-source live \
  --split train \
  --num-examples 10000 \
  --prompt-style none \
  --model Qwen/Qwen2-0.5B-Instruct \
  --output-dir Qwen2-CodebookQA-SFT
```


### 7) GRPO training (TRL)

`scripts/train_grpo_codebook_qa.py` trains GRPO from the live dataloader and supports all prompt styles.

Example:

```bash
python3 scripts/train_grpo_codebook_qa.py \
  --model Qwen/Qwen2-0.5B-Instruct \
  --adapter-model Qwen2-CodebookQA-SFT \
  --output-dir Qwen2-CodebookQA-GRPO \
  --split train \
  --num-examples 1000 \
  --prompt-style none
```

(Use `--prompt-style full` if you want GRPO prompts to always include the long formatting contract.)


### 8) Inference (pattern)

There isn’t a dedicated inference script in `scripts/` yet, but the recommended pattern is:

- Build the same content prompt shape: `Story/Codebook/Question/Assistant:`
- Prepend the same prefix style you trained with (or a stricter one at inference).

Pseudo-code:

```python
from dataloader.trl_adapters import build_task_prefix

prompt = build_task_prefix("abbr") + (
    "Story:\n...\n\nCodebook:\n...\n\nQuestion:\n...\n\nAssistant:\n"
)
# feed prompt to your model.generate(...)
```
